pull data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import pyodbc
# -----------------------------------------------------------------------------
# ANALYSIS PERIOD
# Summer 2026
# -----------------------------------------------------------------------------

analysis_start_date = pd.Timestamp("2026-03-29")
analysis_end_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

arrival_dates = pd.date_range(
    analysis_start_date,
    analysis_end_date,
    freq="D"
)

print(f"Analysis period:")
print(f"Start: {analysis_start_date.date()}")
print(f"End:   {analysis_end_date.date()}")

# -----------------------------------------------------------------------------
# FORECAST FILE LOCATION
# -----------------------------------------------------------------------------

user = 'jamie_douglas'

dsn = 'AzureConnection'
# user = 'christina_k'

username = f'{user}@edinburghairport.com'

connect = pyodbc.connect('DSN=' + dsn + ';')

forecast_folder = Path(
    rf"C:\Users\{user}\Edinburgh Airport Limited\Business Planning - Documents\02. Capacity Areas\09. Car Parking\01. FastPark\01 Short-Term Models\Outputs\Daily Forecast Output"
)

# -----------------------------------------------------------------------------
# LOAD ACTUALS
# -----------------------------------------------------------------------------

# Assumes:
# - connect already exists
# - pyodbc/sqlalchemy connection already created

# We need actuals for the last completed week only

sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [CheckInStarted] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [CheckInStarted] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_entries_df = pd.read_sql(sql, connect)

fastpark_actual_entries_df = fastpark_actual_entries_df.drop_duplicates(subset='BookingReference')


sql = f"""
SELECT *
FROM FastPark.v_EntryAndExits
WHERE [ActualCheckedOutDate] >= '{analysis_start_date.strftime('%Y-%m-%d')}'
    AND [ActualCheckedOutDate] < '{(analysis_end_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')}'
"""

fastpark_actual_exits_df = pd.read_sql(sql, connect)

fastpark_actual_exits_df = fastpark_actual_exits_df.drop_duplicates(subset='BookingReference')

# -----------------------------------------------------------------------------
# PREP ACTUALS
# -----------------------------------------------------------------------------

# Entries
fastpark_actual_entries_df["Date"] = pd.to_datetime(
    fastpark_actual_entries_df["CheckInStarted"]
).dt.date

actual_entries_daily = (
    fastpark_actual_entries_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualEntries")
)

# Exits
fastpark_actual_exits_df["Date"] = pd.to_datetime(
    fastpark_actual_exits_df["ActualCheckedOutDate"]
).dt.date

actual_exits_daily = (
    fastpark_actual_exits_df
    .groupby("Date")
    .size()
    .reset_index(name="ActualExits")
)

# Merge actuals
actuals_daily = pd.merge(
    actual_entries_daily,
    actual_exits_daily,
    on="Date",
    how="outer"
).fillna(0)

actuals_daily["Date"] = pd.to_datetime(actuals_daily["Date"])

print("\nActuals daily:")
print(actuals_daily.head())

# -----------------------------------------------------------------------------
# LEAD TIMES TO CHECK
# -----------------------------------------------------------------------------

lead_times = [42, 35, 28, 21, 14, 7]

# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}
# Results dictionary
entries_accuracy_results = {}
exits_accuracy_results = {}

# NEW
entries_forecast_results = {}
exits_forecast_results = {}

entries_actual_results = {}
exits_actual_results = {}
# -----------------------------------------------------------------------------
# LOOP THROUGH EACH DAY IN LAST WEEK
# -----------------------------------------------------------------------------

for target_date in arrival_dates:

    target_date_only = target_date.date()

    # Get actuals
    actual_row = actuals_daily[
        actuals_daily["Date"].dt.date == target_date_only
    ]

    if actual_row.empty:
        print(f"\nNo actuals found for {target_date_only}")
        continue

    actual_entries = actual_row["ActualEntries"].iloc[0]
    actual_exits = actual_row["ActualExits"].iloc[0]

    entries_accuracy_results[target_date_only] = {}
    exits_accuracy_results[target_date_only] = {}
    
    entries_forecast_results[target_date_only] = {}
    exits_forecast_results[target_date_only] = {}

    entries_actual_results[target_date_only] = {}
    exits_actual_results[target_date_only] = {}
    # -------------------------------------------------------------------------
    # CHECK EACH LEAD TIME
    # -------------------------------------------------------------------------

    for lead in lead_times:

        # Forecast file date
        forecast_run_date = target_date - pd.Timedelta(days=lead)

        if forecast_run_date.strftime('%Y-%m-%d') == '2026-06-22':
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast - Copy.csv"
            )
        else:
            forecast_filename = (
                f"{forecast_run_date.strftime('%Y-%m-%d')}_daily_forecast.csv"
            )

        forecast_path = forecast_folder / forecast_filename

        print(f"\nChecking forecast file:")
        print(f"Target date: {target_date_only}")
        print(f"Lead time: {lead}")
        print(f"Forecast file: {forecast_filename}")

        # ---------------------------------------------------------------------
        # FILE EXISTS?
        # ---------------------------------------------------------------------

        if not forecast_path.exists():
            print("Forecast file NOT found - skipping")
            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan
            continue

        try:

            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------
            # -----------------------------------------------------------------
            # LOAD FORECAST FILE
            # -----------------------------------------------------------------

            forecast_df = pd.read_csv(forecast_path)

            print(f"Loaded rows: {len(forecast_df)}")

            # -----------------------------------------------------------------
            # HANDLE POSSIBLE MISSING / DIFFERENT COLUMN NAMES
            # -----------------------------------------------------------------

            # Strip spaces from column names first
            forecast_df.columns = [str(c).strip() for c in forecast_df.columns]

            print("Original columns:")
            print(forecast_df.columns.tolist())

            # -------------------------------------------------------------
            # HANDLE DATE COLUMN
            # -------------------------------------------------------------

            if "Date" not in forecast_df.columns:

                print("Date column header missing - renaming first column to 'Date'")

                cols = list(forecast_df.columns)
                cols[0] = "Date"

                forecast_df.columns = cols

            # -------------------------------------------------------------
            # HANDLE ENTRIES COLUMN
            # -------------------------------------------------------------

            entries_col = None

            possible_entries_cols = [
                "Entries",
                "entry_forecast"
            ]

            for col in possible_entries_cols:
                if col in forecast_df.columns:
                    entries_col = col
                    break

            if entries_col is None:
                print("No entries forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -------------------------------------------------------------
            # HANDLE EXITS COLUMN
            # -------------------------------------------------------------

            exits_col = None

            possible_exits_cols = [
                "Exits",
                "exit_forecast"
            ]

            for col in possible_exits_cols:
                if col in forecast_df.columns:
                    exits_col = col
                    break

            if exits_col is None:
                print("No exits forecast column found")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            print(f"Using entries column: {entries_col}")
            print(f"Using exits column:   {exits_col}")

            # -----------------------------------------------------------------
            # CLEAN DATE
            # -----------------------------------------------------------------
            date_sample = str(forecast_df["Date"].dropna().iloc[0])

            print(f"Sample date value: {date_sample}")

            # Detect format automatically
            if "-" in date_sample:
                
                # Likely YYYY-MM-DD
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%Y-%m-%d",
                    errors="coerce"
                )

            else:
                
                # Likely DD/MM/YYYY
                forecast_df["Date"] = pd.to_datetime(
                    forecast_df["Date"],
                    format="%d/%m/%Y",
                    errors="coerce"
                )

            # Debugging
            if forecast_df["Date"].isna().all():
                print("All forecast dates failed to parse")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue
            # -----------------------------------------------------------------
            # FILTER TO TARGET DATE
            # -----------------------------------------------------------------

            target_forecast = forecast_df[
                forecast_df["Date"].dt.date == target_date_only
            ]

            if target_forecast.empty:
                print("No matching forecast rows for target date")
                entries_accuracy_results[target_date_only][lead] = np.nan
                exits_accuracy_results[target_date_only][lead] = np.nan
                continue

            # -----------------------------------------------------------------
            # DAILY FORECAST TOTALS
            # -----------------------------------------------------------------

            forecast_entries = pd.to_numeric(
                target_forecast[entries_col],
                errors="coerce"
            ).fillna(0).sum()

            forecast_exits = pd.to_numeric(
                target_forecast[exits_col],
                errors="coerce"
            ).fillna(0).sum()

            print(f"Forecast Entries: {forecast_entries:.2f}")
            print(f"Forecast Exits:   {forecast_exits:.2f}")

            print(f"Actual Entries:   {actual_entries}")
            print(f"Actual Exits:     {actual_exits}")

            # Store forecasts
            entries_forecast_results[target_date_only][lead] = forecast_entries
            exits_forecast_results[target_date_only][lead] = forecast_exits

            # Store actuals
            entries_actual_results[target_date_only][lead] = actual_entries
            exits_actual_results[target_date_only][lead] = actual_exits


            # -----------------------------------------------------------------
            # ACCURACY CALC
            # Formula:
            # (Actual - Forecast) / Actual
            # -----------------------------------------------------------------

            if actual_entries != 0:
                entries_accuracy = 1-abs((
                    (actual_entries - forecast_entries)
                    / actual_entries
                ))
            else:
                entries_accuracy = np.nan

            if actual_exits != 0:
                exits_accuracy = 1-abs((
                    (actual_exits - forecast_exits)
                    / actual_exits
                ))
            else:
                exits_accuracy = np.nan

            entries_accuracy_results[target_date_only][lead] = entries_accuracy
            exits_accuracy_results[target_date_only][lead] = exits_accuracy

            print(f"Entries Accuracy: {entries_accuracy:.4f}")
            print(f"Exits Accuracy:   {exits_accuracy:.4f}")

        except Exception as e:

            print(f"ERROR reading forecast file:")
            print(f"{forecast_filename}")
            print(str(e))

            entries_accuracy_results[target_date_only][lead] = np.nan
            exits_accuracy_results[target_date_only][lead] = np.nan

# -----------------------------------------------------------------------------
# FINAL DATAFRAMES
# -----------------------------------------------------------------------------

entries_accuracy_df = pd.DataFrame.from_dict(
    entries_accuracy_results,
    orient="index"
)

exits_accuracy_df = pd.DataFrame.from_dict(
    exits_accuracy_results,
    orient="index"
)

entries_accuracy_df.index.name = "Date"
exits_accuracy_df.index.name = "Date"

entries_accuracy_df = entries_accuracy_df.sort_index()
exits_accuracy_df = exits_accuracy_df.sort_index()

lead_name_map = {
    42: "6_weeks_out",
    35: "5_weeks_out",
    28: "4_weeks_out",
    21: "3_weeks_out",
    14: "2_weeks_out",
    7: "1_week_out"
}

# Rename columns
entries_accuracy_df.columns = [
    lead_name_map[c]
    for c in entries_accuracy_df.columns
]

exits_accuracy_df.columns = [
    lead_name_map[c]
    for c in exits_accuracy_df.columns
]

# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------

print("\n================ ENTRIES ACCURACY ================\n")
print(entries_accuracy_df)

print("\n================ EXITS ACCURACY ================\n")
print(exits_accuracy_df)

# -----------------------------------------------------------------------------
# FORECAST VALUE TABLES
# -----------------------------------------------------------------------------

entries_forecast_df = pd.DataFrame.from_dict(
    entries_forecast_results,
    orient="index"
)

exits_forecast_df = pd.DataFrame.from_dict(
    exits_forecast_results,
    orient="index"
)

entries_forecast_df.index.name = "Date"
exits_forecast_df.index.name = "Date"

entries_forecast_df = entries_forecast_df.sort_index()
exits_forecast_df = exits_forecast_df.sort_index()

entries_forecast_df.columns = [
    lead_name_map[c]
    for c in entries_forecast_df.columns
]

exits_forecast_df.columns = [
    lead_name_map[c]
    for c in exits_forecast_df.columns
]

# -----------------------------------------------------------------------------
# ACTUAL VALUE TABLES
# -----------------------------------------------------------------------------

entries_actual_df = pd.DataFrame.from_dict(
    entries_actual_results,
    orient="index"
)

exits_actual_df = pd.DataFrame.from_dict(
    exits_actual_results,
    orient="index"
)

entries_actual_df.index.name = "Date"
exits_actual_df.index.name = "Date"

entries_actual_df = entries_actual_df.sort_index()
exits_actual_df = exits_actual_df.sort_index()

entries_actual_df.columns = [
    lead_name_map[c] for c in entries_actual_df.columns
]

exits_actual_df.columns = [
    lead_name_map[c] for c in exits_actual_df.columns
]

# -----------------------------------------------------------------------------
# PRINT OUTPUTS
# -----------------------------------------------------------------------------

print("\n================ ENTRIES FORECASTS ================\n")
print(entries_forecast_df)

print("\n================ EXITS FORECASTS ================\n")
print(exits_forecast_df)

print("\n================ ENTRIES ACTUALS ================\n")
print(entries_actual_df)

print("\n================ EXITS ACTUALS ================\n")
print(exits_actual_df)

# -----------------------------------------------------------------------------
# MONTHLY ACCURACY SUMMARY
# -----------------------------------------------------------------------------

entries_monthly = entries_accuracy_df.copy()
entries_monthly.index = pd.to_datetime(entries_monthly.index)

entries_monthly["Month"] = (
    entries_monthly.index.to_period("M")
)

monthly_entries_accuracy = (
    entries_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY ENTRIES ACCURACY ================\n")
print((monthly_entries_accuracy * 100).round(1))


exits_monthly = exits_accuracy_df.copy()
exits_monthly.index = pd.to_datetime(exits_monthly.index)

exits_monthly["Month"] = (
    exits_monthly.index.to_period("M")
)

monthly_exits_accuracy = (
    exits_monthly
    .groupby("Month")
    .mean()
)

print("\n================ MONTHLY EXITS ACCURACY ================\n")
print((monthly_exits_accuracy * 100).round(1))

Analysis period:
Start: 2026-03-29
End:   2026-10-29


C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_3812\2511526661.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_entries_df = pd.read_sql(sql, connect)
C:\Users\jamie_douglas\AppData\Local\Temp\ipykernel_3812\2511526661.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fastpark_actual_exits_df = pd.read_sql(sql, connect)



Actuals daily:
        Date  ActualEntries  ActualExits
0 2026-03-29            407          434
1 2026-03-30            418          449
2 2026-03-31            338          363
3 2026-04-01            530          334
4 2026-04-02            596          292

Checking forecast file:
Target date: 2026-03-29
Lead time: 42
Forecast file: 2026-02-15_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_forecast', 'exit_forecast']
Date column header missing - renaming first column to 'Date'
Using entries column: entry_forecast
Using exits column:   exit_forecast
Sample date value: 2026-02-16
Forecast Entries: 493.00
Forecast Exits:   564.00
Actual Entries:   407
Actual Exits:     434
Entries Accuracy: 0.7887
Exits Accuracy:   0.7005

Checking forecast file:
Target date: 2026-03-29
Lead time: 35
Forecast file: 2026-02-22_daily_forecast.csv
Loaded rows: 58
Original columns:
['Unnamed: 0', 'booking_entries', 'booking_exits', 'entry_fo

save to excel

In [2]:
# -----------------------------------------------------------------------------
# SAVE TO EXCEL
# -----------------------------------------------------------------------------

output_folder = Path(
    rf"C:\Users\{user}\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / (
    f"forecast_accuracy_{pd.Timestamp.today().strftime('%d%m%y')}.xlsx"
)

# -----------------------------------------------------------------------------
# WRITE TO EXCEL
# -----------------------------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    sheet_name = "Forecast Accuracy"

    # -------------------------------------------------------------------------
    # ENTRIES SECTION
    # -------------------------------------------------------------------------

    current_row = 0

    entries_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet = writer.sheets[sheet_name]

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Forecast"

    current_row += len(entries_forecast_df) + 4

    entries_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Actuals"

    current_row += len(entries_actual_df) + 4

    entries_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=0
    )

    worksheet.cell(
        row=current_row + 1,
        column=1
    ).value = "Entries Accuracy"

    # -------------------------------------------------------------------------
    # EXITS SECTION
    # -------------------------------------------------------------------------

    exits_start_col = len(entries_forecast_df.columns) + 4

    current_row = 0

    exits_forecast_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Forecast"

    current_row += len(exits_forecast_df) + 4

    exits_actual_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Actuals"

    current_row += len(exits_actual_df) + 4

    exits_accuracy_df.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=current_row,
        startcol=exits_start_col
    )

    worksheet.cell(
        row=current_row + 1,
        column=exits_start_col + 1
    ).value = "Exits Accuracy"

    # -------------------------------------------------------------------------
    # MONTHLY SUMMARY SHEETS
    # -------------------------------------------------------------------------

    monthly_entries_accuracy.to_excel(
        writer,
        sheet_name="Monthly Entries Accuracy"
    )

    monthly_exits_accuracy.to_excel(
        writer,
        sheet_name="Monthly Exits Accuracy"
    )

    monthly_entry_ws = writer.sheets["Monthly Entries Accuracy"]
    monthly_exit_ws = writer.sheets["Monthly Exits Accuracy"]

    # -------------------------------------------------------------------------
    # FORMATTING
    # -------------------------------------------------------------------------

    from openpyxl.styles import Font

    bold_font = Font(bold=True)

    # Main sheet titles
    for row in worksheet.iter_rows():
        for cell in row:
            if cell.value in [
                "Entries Forecast",
                "Entries Actuals",
                "Entries Accuracy",
                "Exits Forecast",
                "Exits Actuals",
                "Exits Accuracy"
            ]:
                cell.font = bold_font

    # Monthly sheet titles
    monthly_entry_ws["A1"].font = bold_font
    monthly_exit_ws["A1"].font = bold_font

    # Monthly sheets as %
    for ws in [monthly_entry_ws, monthly_exit_ws]:

        for row in ws.iter_rows(
            min_row=2,
            min_col=2
        ):
            for cell in row:

                if isinstance(cell.value, (int, float)):
                    cell.number_format = "0.0%"

    # Auto-width all sheets
    for ws in [worksheet, monthly_entry_ws, monthly_exit_ws]:

        for column_cells in ws.columns:

            length = 0
            column = column_cells[0].column_letter

            for cell in column_cells:

                try:
                    length = max(
                        length,
                        len(str(cell.value))
                    )
                except:
                    pass

            ws.column_dimensions[column].width = length + 3

print(f"\nExcel saved:")
print(output_file)


Excel saved:
C:\Users\jamie_douglas\OneDrive - Edinburgh Airport Limited\Documents\Fast Park Report\Summer Wash-up 2026\forecast_accuracy_301026.xlsx


## Forecast Accuracy Visual Story (Slide-Ready)
These visuals are structured to answer:
1. How accuracy evolves from 6 weeks out to 1 week out, month by month.
2. What factors are most associated with inaccuracy.
3. How June compares with September and October.

In [28]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------------------------------------------------------
# VISUAL CONFIG
# -----------------------------------------------------------------------------

FONT_FAMILY = "Aptos, Segoe UI, Arial"
BASE_LAYOUT = dict(
    template="simple_white",
    font=dict(family=FONT_FAMILY, size=14, color="#1F2937"),
    title_font=dict(size=24, color="#0F172A"),
    margin=dict(l=60, r=30, t=90, b=60),
    plot_bgcolor="#FAFAF8",
    paper_bgcolor="#FAFAF8"
 )

LEAD_ORDER = [
    "6_weeks_out",
    "5_weeks_out",
    "4_weeks_out",
    "3_weeks_out",
    "2_weeks_out",
    "1_week_out"
]

LEAD_LABELS = {
    "6_weeks_out": "6W",
    "5_weeks_out": "5W",
    "4_weeks_out": "4W",
    "3_weeks_out": "3W",
    "2_weeks_out": "2W",
    "1_week_out": "1W"
}

MONTH_HIGHLIGHTS = ["2026-06", "2026-09", "2026-10"]
HIGHLIGHT_COLORS = {
    "2026-06": "#0B6E4F",
    "2026-09": "#D95F02",
    "2026-10": "#6A4C93"
}

def monthly_accuracy_long(df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    tmp = df.copy()
    if isinstance(tmp.index, pd.PeriodIndex):
        tmp.index = tmp.index.to_timestamp()
    else:
        tmp.index = pd.to_datetime(tmp.index)
    ordered_cols = [c for c in LEAD_ORDER if c in tmp.columns]
    tmp = tmp[ordered_cols]
    long_df = (
        tmp.reset_index()
        .rename(columns={"index": "Month"})
        .melt(id_vars="Month", var_name="Lead", value_name="Accuracy")
    )
    long_df["Month"] = pd.to_datetime(long_df["Month"]).dt.to_period("M").astype(str)
    long_df["LeadLabel"] = long_df["Lead"].map(LEAD_LABELS)
    long_df["AccuracyPct"] = long_df["Accuracy"] * 100
    long_df["Metric"] = metric_name
    return long_df.dropna(subset=["AccuracyPct"])

entries_monthly_long = monthly_accuracy_long(monthly_entries_accuracy, "Entries")
exits_monthly_long = monthly_accuracy_long(monthly_exits_accuracy, "Exits")
monthly_long = pd.concat([entries_monthly_long, exits_monthly_long], ignore_index=True)

available_months = sorted(monthly_long["Month"].unique().tolist())
available_highlights = [m for m in MONTH_HIGHLIGHTS if m in available_months]

print("Prepared monthly tables for visualisation.")
print(f"Months available: {available_months}")
print(f"Highlighted months used: {available_highlights}")

Prepared monthly tables for visualisation.
Months available: ['2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10']
Highlighted months used: ['2026-06', '2026-09', '2026-10']


### 1) Accuracy Progression: 6 Weeks Out to 1 Week Out (Monthly Context)

In [29]:
plot_df = monthly_long.copy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries", "Exits"),
    shared_yaxes=True
)

for col_idx, metric in enumerate(["Entries", "Exits"], start=1):
    metric_df = plot_df[plot_df["Metric"] == metric]

    all_months = sorted(metric_df["Month"].unique().tolist())
    baseline_months = [m for m in all_months if m not in available_highlights]

    for month in baseline_months:
        part = metric_df[metric_df["Month"] == month]
        fig.add_trace(
            go.Scatter(
                x=part["LeadLabel"],
                y=part["AccuracyPct"],
                mode="lines+markers",
                line=dict(color="#CBD5E1", width=1.5),
                marker=dict(size=6, color="#CBD5E1"),
                name=f"{month} (context)",
                legendgroup="context",
                showlegend=(col_idx == 1 and month == baseline_months[0])
            ),
            row=1,
            col=col_idx
        )

    for month in available_highlights:
        part = metric_df[metric_df["Month"] == month]
        if part.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=part["LeadLabel"],
                y=part["AccuracyPct"],
                mode="lines+markers",
                line=dict(color=HIGHLIGHT_COLORS[month], width=4),
                marker=dict(size=10, color=HIGHLIGHT_COLORS[month]),
                name=month,
                legendgroup=month,
                showlegend=(col_idx == 1)
            ),
            row=1,
            col=col_idx
        )

fig.update_layout(
    **BASE_LAYOUT,
    title="Forecast Accuracy Progression by Lead Time (Monthly)",
    height=560,
    legend_title_text="Month",
    hovermode="x unified"
 )

fig.update_xaxes(
    categoryorder="array",
    categoryarray=[LEAD_LABELS[c] for c in LEAD_ORDER],
    title_text="Forecast Version (Weeks Before Arrival)"
 )
fig.update_yaxes(title_text="Accuracy (%)", range=[55, 100])

fig.show()

### 2) Monthly Heatmap: Where Accuracy Is Strong or Weak

In [30]:
def build_heatmap_matrix(monthly_df: pd.DataFrame) -> pd.DataFrame:
    tmp = monthly_df.copy()
    if isinstance(tmp.index, pd.PeriodIndex):
        tmp.index = tmp.index.astype(str)
    else:
        tmp.index = pd.to_datetime(tmp.index).to_period("M").astype(str)
    ordered_cols = [c for c in LEAD_ORDER if c in tmp.columns]
    tmp = (tmp[ordered_cols] * 100).rename(columns=LEAD_LABELS)
    return tmp

entries_heat = build_heatmap_matrix(monthly_entries_accuracy)
exits_heat = build_heatmap_matrix(monthly_exits_accuracy)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries Accuracy", "Exits Accuracy"),
    horizontal_spacing=0.12
)

fig.add_trace(
    go.Heatmap(
        z=entries_heat.values,
        x=entries_heat.columns,
        y=entries_heat.index,
        colorscale=[
            [0.0, "#B23A48"],
            [0.5, "#F4C95D"],
            [1.0, "#2A9D8F"]
        ],
        zmin=60,
        zmax=100,
        text=np.round(entries_heat.values, 1),
        texttemplate="%{text}",
        textfont=dict(size=11),
        colorbar=dict(title="Accuracy %")
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Heatmap(
        z=exits_heat.values,
        x=exits_heat.columns,
        y=exits_heat.index,
        colorscale=[
            [0.0, "#B23A48"],
            [0.5, "#F4C95D"],
            [1.0, "#2A9D8F"]
        ],
        zmin=60,
        zmax=100,
        text=np.round(exits_heat.values, 1),
        texttemplate="%{text}",
        textfont=dict(size=11),
        showscale=False
    ),
    row=1,
    col=2
)

fig.update_layout(
    **BASE_LAYOUT,
    title="Monthly Accuracy Heatmap by Forecast Version",
    height=560
)

fig.update_xaxes(title_text="Forecast Version")
fig.update_yaxes(title_text="Arrival Month")

fig.show()

### 3) June vs September vs October: Benchmark and Variability

In [31]:
focus_leads = ["6_weeks_out", "3_weeks_out", "1_week_out"]

focus_df = monthly_long[
    monthly_long["Month"].isin(available_highlights)
    & monthly_long["Lead"].isin(focus_leads)
]

summary = (
    focus_df.groupby(["Metric", "Month", "LeadLabel"], as_index=False)["AccuracyPct"]
    .mean()
    .rename(columns={"AccuracyPct": "MeanAccuracy"})
)

# Build daily long-format accuracy directly from accuracy tables
def build_daily_accuracy_long(accuracy_df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    tmp = accuracy_df.copy()
    tmp.index = pd.to_datetime(tmp.index)
    cols = [c for c in LEAD_ORDER if c in tmp.columns]
    out = (
        tmp[cols]
        .reset_index()
        .rename(columns={"index": "Date"})
        .melt(id_vars="Date", var_name="Lead", value_name="Accuracy")
    )
    out["Month"] = out["Date"].dt.to_period("M").astype(str)
    out["LeadLabel"] = out["Lead"].map(LEAD_LABELS)
    out["AccuracyPct"] = out["Accuracy"] * 100
    out["Metric"] = metric_name
    return out.dropna(subset=["AccuracyPct"])

entries_daily_accuracy = build_daily_accuracy_long(entries_accuracy_df, "Entries")
exits_daily_accuracy = build_daily_accuracy_long(exits_accuracy_df, "Exits")
daily_accuracy_long = pd.concat([entries_daily_accuracy, exits_daily_accuracy], ignore_index=True)

daily_focus = daily_accuracy_long[
    daily_accuracy_long["Month"].isin(available_highlights)
    & daily_accuracy_long["Lead"].isin(focus_leads)
]
variability = (
    daily_focus.groupby(["Metric", "Month", "LeadLabel"], as_index=False)["AccuracyPct"]
    .std()
    .rename(columns={"AccuracyPct": "StdDev"})
)

summary = summary.merge(variability, on=["Metric", "Month", "LeadLabel"], how="left")
summary["Label"] = summary["Month"] + " | " + summary["LeadLabel"]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries", "Exits"),
    shared_yaxes=True
)

for i, metric in enumerate(["Entries", "Exits"], start=1):
    part = summary[summary["Metric"] == metric].copy()
    part = part.sort_values(["Month", "LeadLabel"])
    colors = [HIGHLIGHT_COLORS.get(m, "#64748B") for m in part["Month"]]

    fig.add_trace(
        go.Bar(
            x=part["Label"],
            y=part["MeanAccuracy"],
            marker_color=colors,
            error_y=dict(type="data", array=part["StdDev"].fillna(0), visible=True),
            text=part["MeanAccuracy"].round(1),
            textposition="outside",
            name=metric,
            showlegend=False
        ),
        row=1,
        col=i
    )

fig.update_layout(
    **BASE_LAYOUT,
    title="June Benchmark vs September/October Variability",
    height=560
 )

fig.update_yaxes(title_text="Mean Accuracy (%)", range=[55, 100])
fig.update_xaxes(title_text="Month | Forecast Version")

fig.show()

# Distribution view for variability context
fig_box = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Entries Distribution", "Exits Distribution"),
    shared_yaxes=True
)

for i, metric in enumerate(["Entries", "Exits"], start=1):
    metric_part = daily_focus[daily_focus["Metric"] == metric]
    for month in available_highlights:
        month_part = metric_part[metric_part["Month"] == month]
        if month_part.empty:
            continue
        fig_box.add_trace(
            go.Box(
                x=month_part["LeadLabel"],
                y=month_part["AccuracyPct"],
                name=month,
                marker_color=HIGHLIGHT_COLORS[month],
                boxmean=True,
                legendgroup=month,
                showlegend=(i == 1)
            ),
            row=1,
            col=i
        )

fig_box.update_layout(
    **BASE_LAYOUT,
    title="Accuracy Distribution: June vs September vs October",
    height=560,
    boxmode="group"
 )
fig_box.update_yaxes(title_text="Accuracy (%)", range=[40, 100])
fig_box.update_xaxes(title_text="Forecast Version")

fig_box.show()